In [1]:
# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False

Устанавливаем пакет: faiss-cpu
Устанавливаем пакет: sentence-transformers


In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)
print("FAISS доступен:", FAISS_AVAILABLE)

Устройство для работы: cpu
FAISS доступен: True


In [15]:
import os
import pandas as pd

def open_document(doc_id: str, title: str):
    with open(title, 'r', encoding='utf-8') as document:
        text = document.read()
    return {
        "doc_id": doc_id,
        "title": (title[10:-4]),
        "text": text
    }

documents = []
for i, filename in enumerate(os.listdir("documents"), 1):
    doc_id = f"doc_{i:02d}"
    documents.append(open_document(doc_id, os.path.join("documents", filename)))


documents_df = pd.DataFrame(documents)
display(documents_df[["doc_id", "title","text"[:10]]])

benchmark_queries: List[Dict[str, object]] = [
    {
            "query_id": "q01",
            "query": "Какое расширение у шаблонов и обычных документов Microsoft Word?",
            "relevant_doc_ids": ["doc_03"],
        },
    {
        "query_id": "q02",
        "query": "Как добавить или удалить кнопку с панели инструментов в Word?",
        "relevant_doc_ids": ["doc_03"],
    },
    {
        "query_id": "q03",
        "query": "Что такое Loginom Community и для каких целей она создана?",
        "relevant_doc_ids": ["doc_02"],
    },
    {
        "query_id": "q04",
        "query": "Как добавить узел в сценарий?",
        "relevant_doc_ids": ["doc_02"],
    },
    {
        "query_id": "q05",
        "query": "Что означает зелёный и красный контур узла при запуске на обработку?",
        "relevant_doc_ids": ["doc_02"],
    },
    {
        "query_id": "q06",
        "query": "Что делает команда транспонировать в специальной вставке?",
        "relevant_doc_ids": ["doc_01"],
    },
    {
        "query_id": "q07",
        "query": "Как отредактировать данные в ячейке?",
        "relevant_doc_ids": ["doc_01"],
    },
    {
        "query_id": "q08",
        "query": "Как настроить панель быстрого доступа в Excel?",
        "relevant_doc_ids": ["doc_01"],
    }]
benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

,doc_id,title,text
0,doc_01,руководство ecxel,Панель быстрого доступа можно легко изменять и...
1,doc_02,Руководство Loginom,Loginom – Low-code платформа для реализации вс...
2,doc_03,руководство word,Окно программы\nMiсrosoft Word 2000 – текстовы...


,query_id,query,relevant_doc_ids
0,q01,Какое расширение у шаблонов и обычных документ...,[doc_03]
1,q02,Как добавить или удалить кнопку с панели инстр...,[doc_03]
2,q03,Что такое Loginom Community и для каких целей ...,[doc_02]
3,q04,Как добавить узел в сценарий?,[doc_02]
4,q05,Что означает зелёный и красный контур узла при...,[doc_02]
5,q06,Что делает команда транспонировать в специальн...,[doc_01]
6,q07,Как отредактировать данные в ячейке?,[doc_01]
7,q08,Как настроить панель быстрого доступа в Excel?,[doc_01]


Загруженные документы представляют собой фрагменты из руководств пользователя по 3 программам: loginom, excel и word. По выбранной предметной области размуно строить retrieval / mini-RAG, потому что к руководствам чаще всего обращаются с конкретыми вопросами, в них легче всего предположить и найти конкретный ответ по ключевым словам.

In [16]:
# В этом разделе собираем self-contained реализацию:
# чанкинг -> векторизация -> индекс -> поиск -> оценка.

def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks


class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray().astype("float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms


def select_backend(device: str = "cpu") -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(
            model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            device=device,
        )
        print("Используем dense-модель эмбеддингов.")
        return backend
    except Exception as e:
        print("Dense-модель недоступна, переключаемся на fallback.")
        print("Причина:", repr(e))
        return TfidfBackend()


def build_chunks(
    docs: List[Dict[str, str]],
    chunk_size: int,
    overlap: int,
) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_idx, chunk_text_value in enumerate(parts):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
                    "chunk_idx": chunk_idx,
                    "chunk_text": chunk_text_value,
                }
            )
    return rows


@dataclass
class RetrievalArtifacts:
    backend_name: str
    backend: EmbeddingBackend
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object


def build_retriever(
    docs: List[Dict[str, str]],
    chunk_size: int = 40,
    overlap: int = 10,
    device: str = "cpu",
) -> RetrievalArtifacts:
    chunks = build_chunks(docs, chunk_size=chunk_size, overlap=overlap)
    chunks_df = pd.DataFrame(chunks)

    backend = select_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    if not FAISS_AVAILABLE:
        raise RuntimeError("FAISS недоступен. Для этого ноутбука ожидается установленный faiss-cpu.")

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)

    return RetrievalArtifacts(
        backend_name=backend.backend_name,
        backend=backend,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        index=index,
    )


def search_chunks(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype("float32")
    scores, indices = artifacts.index.search(query_vector, top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)


def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "relevant_doc_ids": ", ".join(row["relevant_doc_ids"]),
                "predicted_doc_ids": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "first_relevant_rank": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [18]:

# Собираем baseline-конфигурацию retriever.
baseline_chunk_size = 28
baseline_overlap = 8

artifacts = build_retriever(
    documents,
    chunk_size=baseline_chunk_size,
    overlap=baseline_overlap,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())


for query in benchmark_df["query"]:
    display(Markdown(f"### Запрос: {query}"))
    display(search_chunks(query, artifacts=artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6549.54it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем dense-модель эмбеддингов.
Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 172


,doc_id,title,chunk_id,chunk_idx,chunk_text
0,doc_01,руководство ecxel,doc_01_chunk_0,0,Панель быстрого доступа можно легко изменять и...
1,doc_01,руководство ecxel,doc_01_chunk_1,1,находится в ее правой части и представлена в в...
2,doc_01,руководство ecxel,doc_01_chunk_2,2,рис. 1.1) используются для перемещения по соде...
3,doc_01,руководство ecxel,doc_01_chunk_3,3,"позволяет определить, в каком месте документа ..."
4,doc_01,руководство ecxel,doc_01_chunk_4,4,ярлыки листов Файл Microsoft Excel называется ...


### Запрос: Какое расширение у шаблонов и обычных документов Microsoft Word?

,rank,score,doc_id,title,chunk_text
0,1,0.779979,doc_03,руководство word,"затем шаблон, на основе которого будет создан ..."
1,2,0.764827,doc_03,руководство word,"поле, которое расположен ниже, выбрать (двойны..."
2,3,0.764388,doc_03,руководство word,рис.1. Рис.1 Работа с окнами Многооконная орга...


...

,rank,score,doc_id,title,chunk_text
0,1,0.525644,doc_01,руководство ecxel,между ячейками. Они решаются с использованием ...
1,2,0.453104,doc_01,руководство ecxel,"отпуская ее, растянуть выделение на всю област..."
2,3,0.449629,doc_02,Руководство Loginom,по обработке данных. Шаги задаются узлами из к...


### Запрос: Как отредактировать данные в ячейке?

,rank,score,doc_id,title,chunk_text
0,1,0.656999,doc_01,руководство ecxel,Редактирование данных в ячейке можно выполнить...
1,2,0.602064,doc_01,руководство ecxel,старые данные в ней автоматически заменяются н...
2,3,0.596025,doc_01,руководство ecxel,появятся в ячейке и в строке формул. Для завер...


### Запрос: Как настроить панель быстрого доступа в Excel?

,rank,score,doc_id,title,chunk_text
0,1,0.616741,doc_01,руководство ecxel,и можно выполнять редактирование. В ячейках Ex...
1,2,0.593662,doc_01,руководство ecxel,Панель быстрого доступа можно легко изменять и...
2,3,0.539635,doc_01,руководство ecxel,Microsoft Excel обычно распознает вводимые в я...
